In [61]:
# Import os to manage the current working directory.
import os

# Set the working directory to your MLOps project root.
os.chdir(r"C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality")

# Print the current directory to verify the change.
print("Project directory:", os.getcwd())

Project directory: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality


In [62]:
import pandas as pd
data=pd.read_csv("artifacts\data_ingestion\winequality-red.csv")

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\H_KhaN\AppData\Local\Temp\ipykernel_15432\2925628614.py:2: SyntaxWarning: invalid escape sequence '\d'
  data=pd.read_csv("artifacts\data_ingestion\winequality-red.csv")


In [63]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [64]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [65]:
data.shape

(1599, 12)

In [66]:
data.describe

<bound method NDFrame.describe of       fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0               7.4             0.700         0.00             1.9      0.076   
1               7.8             0.880         0.00             2.6      0.098   
2               7.8             0.760         0.04             2.3      0.092   
3              11.2             0.280         0.56             1.9      0.075   
4               7.4             0.700         0.00             1.9      0.076   
...             ...               ...          ...             ...        ...   
1594            6.2             0.600         0.08             2.0      0.090   
1595            5.9             0.550         0.10             2.2      0.062   
1596            6.3             0.510         0.13             2.3      0.076   
1597            5.9             0.645         0.12             2.0      0.075   
1598            6.0             0.310         0.47             3.6      0.0

In [67]:
# Import Path to work with file and directory paths.
from pathlib import Path

# Import dataclass to create a simple configuration class.
from dataclasses import dataclass


# Define the configuration required by the data validation component.
@dataclass
class Data_validation_Config:

    # Store the directory for validation artifacts.
    root_dir: Path

    # Store the path to the validation status file.
    status_file: Path

    # Store the directory containing the extracted dataset.
    unzip_data_dir: Path

    # Store the expected column names and their schema definitions.
    all_schema: dict

In [69]:
# Import Path to manage filesystem paths.
from pathlib import Path

# Import the configuration dataclass.


# Import helpers that read YAML and create directories.
from src.Mlops_project_quality_predictor.utils.common import read_yaml, create_directories

# Import the default file paths defined in your project.
from src.Mlops_project_quality_predictor.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH


# Define the class responsible for loading project configuration.
class ConfigurationManager:

    # Initialize the configuration manager.
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
    ):

        # Read the main configuration YAML file.
        self.config = read_yaml(config_filepath)

        # Read the model parameters YAML file.
        self.params = read_yaml(params_file_path)

        # Read the dataset schema YAML file.
        self.schema = read_yaml(schema_file_path)

        # Create the main artifacts directory if it does not exist.
        create_directories([self.config.artifacts_root])

    # Build and return the configuration for data validation.
    def get_data_validation(self) -> Data_validation_Config:

        # Get the data validation section using its exact YAML key.
        config = self.config.Data_validation

        # Get the expected column definitions from schema.yaml.
        schema = self.schema.COLUMNS

        # Create the validation output directory if it does not exist.
        create_directories([config.root_dir])

        # Create the configuration object with the required settings.
        data_validation_config = Data_validation_Config(
            root_dir=Path(config.root_dir),  # Convert the output directory to a Path.
            status_file=Path(config.STATUS_FILE),  # Convert the status file path to a Path.
            unzip_data_dir=Path(config.unzip_data_dir),  # Convert the dataset directory to a Path.
            all_schema=dict(schema)  # Convert the schema configuration to a dictionary.
        )

        # Return the completed data validation configuration.
        return data_validation_config

In [70]:

# Import Path to search for CSV files and manage file paths.
from pathlib import Path

# Import pandas to load and inspect the dataset.
import pandas as pd

# Import the configuration dataclass used by this component.



# Define the class that validates the dataset.
class Data_Validation:

    # Initialize the component with its configuration.
    def __init__(self, config: Data_validation_Config):

        # Save the supplied configuration for use in other methods.
        self.config = config

    # Validate whether the dataset contains the expected columns.
    def validation_all_columns(self) -> bool:

        # Get the directory containing the extracted dataset.
        data_dir = Path(self.config.unzip_data_dir)

        # Stop with a useful error if the directory does not exist.
        if not data_dir.is_dir():
            raise FileNotFoundError(
                f"Dataset directory does not exist: {data_dir.resolve()}"
            )

        # Find all CSV files inside the directory and its subdirectories.
        csv_files = list(data_dir.rglob("*.csv"))

        # Stop if no CSV file was found.
        if not csv_files:
            raise FileNotFoundError(
                f"No CSV file found inside: {data_dir.resolve()}"
            )

        # Stop if multiple CSV files exist and the intended file is ambiguous.
        if len(csv_files) > 1:
            raise ValueError(
                f"Multiple CSV files found: {csv_files}. "
                "Please configure the intended dataset file."
            )

        # Select the only CSV file found.
        csv_file = csv_files[0]

        # Read the CSV and automatically detect common delimiters such as commas or semicolons.
        data = pd.read_csv(csv_file, sep=None, engine="python")

        # Get the actual column names from the dataset.
        actual_columns = set(data.columns)

        # Get the expected column names from the schema.
        expected_columns = set(self.config.all_schema.keys())

        # Find expected columns that are missing from the dataset.
        missing_columns = expected_columns - actual_columns

        # Find unexpected columns that are absent from the schema.
        extra_columns = actual_columns - expected_columns

        # Mark validation as successful only when the columns match exactly.
        validation_status = not missing_columns and not extra_columns

        # Create the parent directory for the status file if needed.
        self.config.status_file.parent.mkdir(parents=True, exist_ok=True)

        # Open the status file for writing using UTF-8 encoding.
        with open(self.config.status_file, "w", encoding="utf-8") as file:

            # Record whether validation passed or failed.
            file.write(f"Validation status: {validation_status}\n")

            # Record the path of the dataset that was checked.
            file.write(f"Dataset file: {csv_file.resolve()}\n")

            # Record any expected columns missing from the dataset.
            file.write(f"Missing columns: {sorted(missing_columns)}\n")

            # Record any unexpected columns found in the dataset.
            file.write(f"Unexpected columns: {sorted(extra_columns)}\n")

        # Print the dataset path so you know which file was checked.
        print(f"Dataset checked: {csv_file.resolve()}")

        # Print the validation outcome.
        print(f"Validation completed: {validation_status}")

        # Print the missing columns if any were found.
        print(f"Missing columns: {sorted(missing_columns)}")

        # Print the unexpected columns if any were found.
        print(f"Unexpected columns: {sorted(extra_columns)}")

        # Return the validation result to the calling pipeline.
        return validation_status


In [71]:


# Use try to execute the validation process and catch any errors.
try:

    # Initialize the configuration manager using your YAML files.
    config = ConfigurationManager()

    # Load the configuration required for data validation.
    data_validation_config = config.get_data_validation()

    # Create the data validation object with the loaded configuration.
    data_validation = Data_Validation(config=data_validation_config)

    # Validate the dataset columns and save the validation status.
    validation_status = data_validation.validation_all_columns()

    # Display the final validation result.
    print(f"Validation completed: {validation_status}")

# Catch any exception raised during configuration or validation.
except Exception as e:

    # Print the error type and message to help with debugging.
    print(f"Error: {type(e).__name__}: {e}")

    # Raise the original exception again so Jupyter displays the traceback.
    raise

[2026-10-14 00:22:32,902:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\config\config.yaml]
[2026-10-14 00:22:32,908:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\params.yaml]
[2026-10-14 00:22:32,918:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\schema.yaml]
[2026-10-14 00:22:32,920:INFO:common:created directory at: artifacts]
[2026-10-14 00:22:32,924:INFO:common:created directory at: artifacts/data_validation]
Error: FileNotFoundError: Dataset directory does not exist: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\artifacts\data_ingestion\winequality-red.csv


FileNotFoundError: Dataset directory does not exist: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\artifacts\data_ingestion\winequality-red.csv